In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *
from scipy.ndimage import gaussian_filter
from scipy.signal import convolve2d

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
#files_fdt = sorted(glob.glob('/home/ulyanov/data/cross/*fdt*'))
files_fdt = sorted(glob.glob('/home/ulyanov/data/cross/2024-03-19/*fdt*'))
files_fdt

['/home/ulyanov/data/cross/2024-03-19/solo_L2_phi-fdt-blos_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/2024-03-19/solo_L2_phi-fdt-stokes_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/2024-03-19/solo_L2_phi-fdt-vlos_20240319T152009_V202602220014_0443190639.fits.gz']

In [3]:
#files_hmi = sorted(glob.glob('/home/ulyanov/data/cross/*hmi*'))
files_hmi = sorted(glob.glob('/home/ulyanov/data/cross/2024-03-19/*hmi*'))
files_hmi

['/home/ulyanov/data/cross/2024-03-19/hmi.M_45s.20240319_152445_TAI.2.magnetogram.fits']

In [4]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xu, yu = s['xu'], s['yu']

In [5]:
file_hmi = files_hmi[0]
file_fdt = files_fdt[0]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

binning = header_hmi['RSUN_OBS'] / header_hmi['CDELT1'] / (header_fdt['RSUN_ARC'] / header_fdt['CDELT1'])
print(binning)

#binning *= 2

#data_hmi = gaussian_filter(data_hmi, sigma=binning)
data_hmi = rebin(data_hmi, int(round(binning)), update_header=header_hmi)
#data_hmi = convolve2d(data_hmi, psf, mode='same')
#data_hmi = gaussian_filter(data_hmi, sigma=1)

data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=False, mu_thr=0.1,
                     xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                     crota=header_fdt['CROTA'] + 0.25,
                     grid=crop_grid(xu, yu, header_fdt),
                     kind='quadratic')

data_fdt[np.isnan(data_hmi)] = np.nan

print(np.nanmean(data_hmi[350:400,350:400]), np.nanmean(data_fdt[350:400,350:400]))

3.140790459562361


KeyboardInterrupt: 

In [15]:
ft_fdt = np.fft.fft2(np.nan_to_num(data_fdt))
ft_hmi = np.fft.fft2(np.nan_to_num(data_hmi))

#ft_psf = ft_fdt * np.conj(ft_hmi) / (np.abs(ft_hmi) ** 2 + 1e-4)
ft_psf = ft_hmi * np.conj(ft_fdt) / (np.abs(ft_fdt) ** 2 + 1e-4)
psf = np.real(np.fft.ifft2(ft_psf))
psf = np.roll(psf, (2, 2), axis=(0,1))
psf = psf[:5,:5].clip(0)
psf /= np.sum(psf)

plt.figure(figsize=(10,10))
plt.imshow(psf, cmap='gray')

In [16]:
data_hmi = convolve2d(data_hmi, psf, mode='same')

In [17]:
np.nanmean(data_fdt * data_hmi) / (np.sqrt(np.nanmean(data_fdt ** 2) * np.nanmean(data_hmi ** 2)))

np.float64(0.9417022901010912)

In [18]:
X = rebin(data_hmi, 1)
Y = rebin(data_fdt, 1)

xx, yy = [0], [0]

for q in np.arange(2., 30.25, 0.5):
    w = np.sqrt(X ** 2 + Y ** 2)
    t = np.where(w < q ** 2)
    x, y = X[t], Y[t]
    w = 1#w[t]

    A = np.array([[np.mean(w * x ** 2), np.mean(w * x * y)],
                  [np.mean(w * x * y), np.mean(w * y ** 2)]])

    vals, vecs = np.linalg.eigh(A)
    u, v = vecs[:, 1]
    u, v = u * q ** 2 * np.sign(u), v * q ** 2 * np.sign(u)

    xx = [-u] + xx + [u]
    yy = [-v] + yy + [v]

print(u / v)


xx = np.array(xx)
yy = np.array(yy)

1.0338461175823839


In [19]:
plt.figure(figsize=(10,10))
plt.plot([-400,400], [-400,400], color='black', lw=0.5)
plt.scatter(X, Y, s=0.05)
plt.plot([-u,u],[-v,v], '--', color='black', lw=1.5)
#plt.plot(xx, yy, '--', color='black', lw=1)
plt.plot()

plt.xlabel('HMI, G')
plt.ylabel('FDT, G')

#plt.xlim(-40,40)
#plt.ylim(-40,40)

plt.xlim(-400,400)
plt.ylim(-400,400)



plt.grid(True)
plt.tight_layout()

In [20]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, 'hmimag', vmin=-1000, vmax=1000, origin='lower')#, interpolation='bicubic')
#plt.axis('off')

plt.xlim(300,600)
plt.ylim(300,600)

plt.xlabel('X, pixels')
plt.ylabel('Y, pixels')
plt.title('FDT')

plt.tight_layout()

In [21]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, 'hmimag', vmin=-1000, vmax=1000, origin='lower', interpolation='bilinear')
#plt.axis('off')

plt.xlim(300,600)
plt.ylim(300,600)

plt.xlabel('X, pixels')
plt.ylabel('Y, pixels')
plt.title('HMI')

plt.tight_layout()

In [23]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi - data_fdt, 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()